# Name parts and pair scores

`parse_name` splits a surface into an honorific, a given name, and a surname.

- `Mrs. Harker` is honorific `mrs` plus surname `harker`. `Mr`, `Mrs`, and `Miss` plus one token are surnames. That is the form the married-name rule reads.
- `Count Dracula` and `Madam Mina` keep the single token as a given name. A bare token such as `Jonathan` is also a given name. Recording `Mina` as a surname would scramble the Murray-to-Harker timeline.
- Particles stay on the surname: `Van Helsing`.
- A trailing `'s` is stripped before the tokens are compared.

Clustering does not merge on raw string similarity alone. The weighted score is Jaro-Winkler 0.40, surname agreement 0.30, given agreement 0.20, honorific agreement 0.10. Auto-merge needs 0.85 and support from a shared surname or a rare token. The band from 0.60 to 0.85 is review, not a merge.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "kg").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(ROOT)


In [ ]:
from rapidfuzz.distance import JaroWinkler

from src.kg.resolve import _pair_score, parse_name

WEIGHTS = {"jaro": 0.40, "surname": 0.30, "given": 0.20, "honorific": 0.10}
names = [
    "Count Dracula",
    "Dracula",
    "Castle Dracula",
    "Jonathan",
    "Jonathan Harker",
    "Mrs. Harker",
    "Mina Harker",
    "Van Helsing",
    "Abraham Van Helsing",
]
for name in names:
    part = parse_name(name)
    print(f"{name:24} honorific={part.honorific!s:8} given={part.given!s:12} surname={part.surname}")

pairs = [
    ("Jonathan", "Jonathan Harker"),
    ("Dracula", "Castle Dracula"),
    ("Mina Murray", "Mina Harker"),
    ("Arthur Holmwood", "Lord Godalming"),
    ("Van Helsing", "Abraham Van Helsing"),
]
print()
for left, right in pairs:
    a, b = parse_name(left), parse_name(right)
    jaro = JaroWinkler.normalized_similarity(a.token_str, b.token_str)
    print(f"{left:22} {right:24} jaro={jaro:.3f}  weighted={_pair_score(a, b, WEIGHTS):.3f}")


## How to read those scores

`Jonathan` against `Jonathan Harker` has a high raw Jaro-Winkler because the short string is a prefix of the long one. The weighted score stays under 0.85: the short name has no surname, so the surname term is only the neutral 0.5. The subset rule in the next notebook is what merges them.

`Mina Murray` against `Mina Harker` shares a given name and differs in surname, so the weighted score sits in the review band. A separate rule merges them when one given name is attested with exactly two surnames.

`Dracula` against `Castle Dracula` is a low score, and the two spans have different entity types. Type clash is a cannot-link. `Arthur Holmwood` against `Lord Godalming` is also a low score. That pair merges by comma apposition, or by the hand-written must-link, not by Jaro-Winkler.

A bare `Harker` is a dangerous token. Jonathan and Mina both carry the surname. Exact match on that token must not glue `Mr. Harker` to `Mrs. Harker`. The resolver still sends a bare `Harker` to Jonathan, because Mina married into the name (she is also attested as Mina Murray). In the novel's usage a bare surname is the husband, and Mina is written `Mrs. Harker`, `Madam Mina` or `Mina`. If two men shared the surname, the bare form would stay on its own.
